# 9.5 切 S2 的 softmax 归并：merge 如何保证数值正确

切 S2 买到了并行度，但每核只看到一段 KV，softmax 的**分母是全序列的和**——各核的部分结果必须归并。本节讲清楚 merge 的数学与实现。

## 1. 问题定义

softmax 正常形态：

```
attn(q, K, V) = softmax(qKᵀ / √d) · V
              = (Σ_j exp(q·k_j - m) · v_j) / (Σ_j exp(q·k_j - m))
```

m 是全序列的行最大值（数值稳定项）。切 S2 后核 i 只拿到第 i 段：

```
核 i 可算:  m_i = 段内最大值,  l_i = Σ_段内 exp(q·k_j - ?),  acc_i = Σ_段内 exp(...)·v_j
```

**问号是关键**：每个核都不知道全局 m。merge 要解决的就是把各核的 (m_i, l_i, acc_i) 归并成全局结果。

## 2. 归并公式（与 FA 在线 softmax 同源）

这正是[第2章](../02_fa_principle/README.md)在线 softmax 跨块递推的**核间版本**：

```
全局:  M = max_i(m_i)
       L = Σ_i l_i · exp(m_i - M)
       ACC = Σ_i acc_i · exp(m_i - M)
结果:  attn = ACC / L
```

每核的局部和与部分累加都乘一个**缩放因子 `exp(m_i - M)`**——因为全局最大值变了，原来以 m_i 为基准的指数都要重新对齐。这个「最大值 + 缩放」的套路和[第5章](../05_flash_attn_non_quantized/README.md) softmax update 的行间递推一模一样，只是作用域从「块间」变成「核间」。

## 3. merge kernel 的两种形态

| 形态 | 流程 | 适用 |
|--|--|--|
| **算子内 merge**（kernel 末尾/独立 kernel） | 各核把 (m, l, acc) 写到 GM 的临时 buffer，merge kernel 读入归并 | 通用，FA decode 标准做法 |
| **原子规约** | 各核对全局 (m, l, acc) 做原子 max/add | 实现简单但原子操作性能差，少用 |

工程上选前者：**临时 buffer 按 (核数 × 行数) 组织，merge kernel 是一个纯 Vector 的归并任务**，与本轮 Cube 循环完全解耦。

## 图示

![切 S2 的分核与 merge](images/s2_split_merge.svg)

## 动手实验：验证归并公式的数值正确性


In [ ]:
# 动手: 切 S2 归并公式的数值验证 —— 与不切的 softmax 结果对比
import numpy as np

rng = np.random.default_rng(0)
d, S2, n_seg = 64, 1024, 8            # head 维 / KV 长 / 切成 8 段(8 核)

q = rng.normal(size=(1, d))            # 单行 Q (decode)
K = rng.normal(size=(S2, d))
V = rng.normal(size=(S2, d))

# ---- 参考: 不切的完整 softmax ----
logits = (K @ q.T).ravel() / np.sqrt(d)
ref = np.exp(logits - logits.max()) @ V / np.exp(logits - logits.max()).sum()

# ---- 切 S2: 每核一段, 只见局部数据 ----
seg = S2 // n_seg
m_list, l_list, acc_list = [], [], []
for i in range(n_seg):
    k_seg = K[i*seg:(i+1)*seg]
    v_seg = V[i*seg:(i+1)*seg]
    s = (k_seg @ q.T).ravel() / np.sqrt(d)
    m_i = s.max()                      # 段内最大值
    p = np.exp(s - m_i)                # 段内 softmax 分子
    l_i = p.sum()                      # 段内和
    acc_i = p @ v_seg                  # 段内部分累加
    m_list.append(m_i); l_list.append(l_i); acc_list.append(acc_i)

# ---- merge: 按归并公式合成全局 ----
M = max(m_list)
L = sum(l * np.exp(m - M) for m, l in zip(m_list, l_list))
ACC = sum(acc * np.exp(m - M) for m, acc in zip(m_list, acc_list))
merged = ACC / L

print(f"切 {n_seg} 段 + merge vs 不切, 最大误差: {np.abs(merged - ref).max():.2e}")
print(f">>> 数值与参考一致 —— 段内最大值 m_i 换算成全局 M 的缩放 exp(m_i - M) 是正确性的关键")


### 观察结论

切 8 段归并后的结果与不切的参考实现完全一致——**缩放因子 `exp(m_i - M)` 保证了各段局部统计量可以任意顺序归并**。这也解释了为什么 merge kernel 只需要存 (m, l, acc) 三个量而不是完整的 softmax 分布。

## 4. merge 的性能账

切 S2 不是纯赚，多出来的成本要算进总账：

| 成本项 | 量级 |
|--|--|
| 临时 buffer 读写 | (核数 × 行数 × (1 + 1 + d)) 个元素，GM 两跳 |
| merge kernel 本身 | 行数 × 核数 的归并计算（纯 Vector） |
| 核间同步 | 所有分段核完成才能启动 merge（一个 pipe barrier / 事件） |

经验法则：**段数 = 核数时收益最大；切得比核数细只会让 merge 成本线性上涨而并行度不再增加**。

## 5. 与第5章 update 的统一视角

把三处「最大值缩放」放在一起看，它们是同一个数学结构的三个实例：

| 位置 | 作用域 | 公式 |
|--|--|--|
| 第5章 softmax update | 同核 S2 块之间 | `m_new = max(m_old, m_blk)`，旧 acc 乘 `exp(m_old - m_new)` |
| 9.3/9.4 切 S2 merge | 核之间 | `M = max(m_i)`，各核 acc 乘 `exp(m_i - M)` |
| 第7章均衡性 | （比喻）核间装箱 | max 哨兵核决定总耗时 |

**在线 softmax 的本质就是「局部统计量 + 可交换的归并」**——理解了这一点，块间、核间、甚至多卡间的 attention 归并（如 Ring Attention）都是同一套公式。

## 本节要点

- 切 S2 后每核输出三元组 (m_i, l_i, acc_i)，merge 按 `M = max m_i`、`缩放 exp(m_i - M)` 归并；
- 归并公式与 FA 在线 softmax 的块间 update 同构，是它的核间版本；
- merge kernel 用临时 buffer 形态（纯 Vector 归并），避免原子操作；
- 段数取核数收益最大；merge 成本（buffer 读写 + 归并 + 同步）要计入总账。

## 小测验

1. 写出切 S2 后各核的部分统计量与 merge 归并公式。
2. 缩放因子 `exp(m_i - M)` 为什么不可省略？去掉会怎样？
3. merge kernel 为什么选临时 buffer 形态而不是原子规约？
4. 切 S2 的段数是不是越多越好？成本账怎么算？

>>> 参考答案见 [answer/09.05_answer.txt](answer/09.05_answer.txt)
